In [54]:
import pandas as pd
import numpy as np
import matplotlib as mlt
import seaborn as sns
import openpyxl
import csv
import json

In [55]:
tipscsv = pd.read_csv("./data/raw/tips.csv",encoding="utf-8",header=0)
tipsxlsx = pd.read_excel("./data/raw/tips.xlsx",sheet_name=0)
titancsv = pd.read_csv("./data/raw/titanic.csv",encoding="utf-8")
tipsjson= pd.read_json("./data/raw/tips.json",orient="records")

## 1번

In [56]:
#1
fare = titancsv["fare"]
print(type(fare))
#2
tit = titancsv[["survived","pclass","sex","age"]]
print(type(tit),tit.shape)
#3
#시리즈 타입은 열 하나만을 표기하기에 하나의 열을 반환한 1번은 시리즈타입
#여러 열이 있다면 데이터프레임타입이 된다.

<class 'pandas.Series'>
<class 'pandas.DataFrame'> (891, 4)


## 2번

In [57]:
print(titancsv[["sex","age"]].loc[10:15])
print(titancsv[["sex","age"]].iloc[10:15])
#loc는 이상 이하의 범위 iloc는 이상 미만의 범위

       sex   age
10  female   4.0
11  female  58.0
12    male  20.0
13    male  39.0
14  female  14.0
15  female  55.0
       sex   age
10  female   4.0
11  female  58.0
12    male  20.0
13    male  39.0
14  female  14.0


## 3번

In [58]:
titancsv.iloc[-7:-3]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
884,0,3,male,25.0,0,0,7.050,S,Third,man,True,NaN,Southampton,no,True
885,0,3,female,39.0,0,5,29.125,Q,Third,woman,False,NaN,Queenstown,no,False
886,0,2,male,27.0,0,0,13.000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.000,S,First,woman,False,B,Southampton,yes,True


## 4번

In [59]:
mask = titancsv["age"]<10
#1
print(titancsv[mask].head(5))
print()
#2
print("승객 수 : ",mask.sum())

    survived  pclass     sex  age  sibsp  parch     fare embarked   class  \
7          0       3    male  2.0      3      1  21.0750        S   Third   
10         1       3  female  4.0      1      1  16.7000        S   Third   
16         0       3    male  2.0      4      1  29.1250        Q   Third   
24         0       3  female  8.0      3      1  21.0750        S   Third   
43         1       2  female  3.0      1      2  41.5792        C  Second   

      who  adult_male deck  embark_town alive  alone  
7   child       False  NaN  Southampton    no  False  
10  child       False    G  Southampton   yes  False  
16  child       False  NaN   Queenstown    no  False  
24  child       False  NaN  Southampton    no  False  
43  child       False  NaN    Cherbourg   yes  False  

승객 수 :  62


## 5번

In [60]:
#1
cla = (titancsv["class"]=="Third") & (titancsv["sex"]=="male") & (titancsv["survived"])
print(len(titancsv[cla]))
print()
#2
cla2 = (titancsv["class"]=="First") |(titancsv["fare"]>=100)
print(len(titancsv[cla2]))
print()
#3
cla3 = titancsv["alone"]
print(len(titancsv[cla3]))

47

216

537


## 6번

In [61]:
#1
area = titancsv["embark_town"].isin(["Cherbourg","Queenstown"])
print(titancsv[area].shape)
#2
age = titancsv["age"].between(20,30,inclusive="both")
age2 = titancsv["age"].between(20,30,inclusive="neither")
print(titancsv["age"].loc[age].min())
print(titancsv["age"].loc[age].max())
print(titancsv["age"].loc[age].shape)
#3
print(titancsv["age"].loc[age2].shape)



(245, 15)
20.0
30.0
(245,)
(205,)


## 7번

In [62]:
mask2 = (titancsv["age"]<=12) & (titancsv["class"]=="Third")
kids = titancsv[["age","sex","survived"]].loc[mask2]
print(len(kids))
print(((kids["survived"].sum()/len(kids))*100).round(2))

48
41.67


## 8번

In [63]:
min_fare = 50
mask3 = titancsv.query(f"embark_town == 'Cherbourg' and fare >= {min_fare}")
print(mask3.shape)

(61, 15)


## 9번

In [64]:
sort = titancsv[["pclass","fare","sex","age"]].sort_values(by=["pclass","fare"],ascending=[True,False])
print(sort.head(3))
print(sort.loc[0])
print(sort.iloc[0])
#loc는 인덱스 값이 0인것 iloc는 현재 정렬된 데이터프레임의  0번째 행

     pclass      fare     sex   age
258       1  512.3292  female  35.0
679       1  512.3292    male  36.0
737       1  512.3292    male  35.0
pclass       3
fare      7.25
sex       male
age       22.0
Name: 0, dtype: object
pclass           1
fare      512.3292
sex         female
age           35.0
Name: 258, dtype: object


## 10번

In [65]:
mask4 = (titancsv["sex"]=="male") & (titancsv["age"].between(20,40,inclusive="both")) & (titancsv["class"].isin(["First","Second"]))
print("행 : ", len(titancsv[mask4]))
print("생존율 :",((titancsv["survived"].loc[mask4].sum()/len(titancsv[mask4]))*100).round(2))
print("요금 상위 3명 : ",titancsv[mask4].sort_values("fare",ascending=False).head(3))
query = titancsv.query( "sex == 'male' and 20 <= age <= 40 and (`class` == 'First' or `class` == 'Second')")
print(len(query))
class4 = titancsv.query( "sex == 'male' and 20 <= age <= 40 and (`class` == 'Third')")
print(((class4["survived"].sum()/len(class4))*100).round(2))

행 :  103
생존율 : 24.27
요금 상위 3명 :       survived  pclass   sex   age  sibsp  parch      fare embarked  class  \
737         1       1  male  35.0      0      0  512.3292        C  First   
679         1       1  male  36.0      0      1  512.3292        C  First   
118         0       1  male  24.0      0      1  247.5208        C  First   

     who  adult_male deck embark_town alive  alone  
737  man        True    B   Cherbourg   yes   True  
679  man        True    B   Cherbourg   yes  False  
118  man        True    B   Cherbourg    no  False  
103
15.09


일단 1등석과 2등석을 합쳐서 계산한 생존율이 3등석보다는 월등히 높다. 아마 고급 객실에 있었을 수록 조금더 빠르게 구명정으로 안내받을 기회가 있지 않았을까라는 생각이 든다.